<a href="https://colab.research.google.com/github/vaideheedaf/ML-Coral-reef/blob/main/Naive_bayes_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)



In [ ]:

train = pd.read_csv("ML_train.csv")
valid = pd.read_csv("ML_valid.csv")
test  = pd.read_csv("ML_test.csv")

FEATURES = ["mean_saturation", "mean_brightness", "white_pixel_pct", "mean_lightness",
            "lbp_mean", "glcm_contrast", "glcm_homogeneity", "edge_density", "entropy"]

X_train, y_train = train[FEATURES], train["target"]
X_valid, y_valid = valid[FEATURES], valid["target"]
X_test,  y_test  = test[FEATURES],  test["target"]

In [ ]:

base = GaussianNB().fit(X_train, y_train)
print(f"Baseline GaussianNB  valid acc: {accuracy_score(y_valid, base.predict(X_valid)):.4f}")


Baseline GaussianNB  valid acc: 0.7603


In [ ]:

results = []
for use_power in [False, True]:
    for vs in [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1]:
        steps = ([("power", PowerTransformer())] if use_power else []) + [("nb", GaussianNB(var_smoothing=vs))]
        m = Pipeline(steps).fit(X_train, y_train)
        acc = accuracy_score(y_valid, m.predict(X_valid))
        results.append((acc, use_power, vs))
        print(f"PowerTransformer={use_power!s:5}  var_smoothing={vs:<7}  valid acc={acc:.4f}")

best_acc, best_power, best_vs = max(results)
print(f"\nBest on valid: acc={best_acc:.4f}  PowerTransformer={best_power}  var_smoothing={best_vs}")


PowerTransformer=False  var_smoothing=1e-09    valid acc=0.7603
PowerTransformer=False  var_smoothing=1e-07    valid acc=0.7581
PowerTransformer=False  var_smoothing=1e-05    valid acc=0.7538
PowerTransformer=False  var_smoothing=0.001    valid acc=0.7948
PowerTransformer=False  var_smoothing=0.01     valid acc=0.8251
PowerTransformer=False  var_smoothing=0.1      valid acc=0.6739
PowerTransformer=True   var_smoothing=1e-09    valid acc=0.8035
PowerTransformer=True   var_smoothing=1e-07    valid acc=0.8035
PowerTransformer=True   var_smoothing=1e-05    valid acc=0.8035
PowerTransformer=True   var_smoothing=0.001    valid acc=0.8035
PowerTransformer=True   var_smoothing=0.01     valid acc=0.8035
PowerTransformer=True   var_smoothing=0.1      valid acc=0.8056

Best on valid: acc=0.8251  PowerTransformer=False  var_smoothing=0.01


In [ ]:

steps = ([("power", PowerTransformer())] if best_power else []) + [("nb", GaussianNB(var_smoothing=best_vs))]
final = Pipeline(steps).fit(X_train, y_train)

pred = final.predict(X_test)
print("\n=== TEST RESULTS ===")
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, digits=4))
print("Confusion matrix (rows=true, cols=pred; order Bleached, Healthy):")
print(confusion_matrix(y_test, pred, labels=["Bleached", "Healthy"]))


=== TEST RESULTS ===
Accuracy: 0.821
              precision    recall  f1-score   support

    Bleached     0.8504    0.8000    0.8244       135
     Healthy     0.7923    0.8443    0.8175       122

    accuracy                         0.8210       257
   macro avg     0.8214    0.8221    0.8209       257
weighted avg     0.8228    0.8210    0.8211       257

Confusion matrix (rows=true, cols=pred; order Bleached, Healthy):
[[108  27]
 [ 19 103]]
